# Phase 4 — Model Evaluation

## Overview
This notebook is the academic core of the project. It produces all evaluation evidence required for the capstone report:

1. **Metric comparison table** across all 5 models
2. **Target benchmarks** check (MAE < 0.3pp, RMSE < 0.5pp, etc.)
3. **SHAP explainability** — which features drive the XGBoost predictions
4. **Residual analysis** — is the best model's error pattern random or systematic?
5. **Walk-forward validation** — 177 expanding-window predictions on 2011–2025
6. **Ablation study** — removing feature groups one at a time
7. **Diebold-Mariano test** — statistical proof the best model is significantly better than ARIMA
8. **SPF comparison** — our ensemble vs. U-Michigan professional forecasters

### Headline results (2019–2025 test set)
| Model | MAE | RMSE | R² | Dir. Acc |
|-------|-----|------|----|----------|
| Ensemble (ours) | **0.84** | **1.22** | **0.72** | 60% |
| XGBoost | 1.53 | 2.14 | 0.14 | 72.5% |
| Linear Regression | 1.35 | 2.14 | 0.14 | 81.3% |
| LSTM | 1.84 | 2.47 | -0.14 | 55% |
| ARIMA | 1.96 | 2.83 | -0.49 | 50% |

The ensemble achieves **R² = 0.72** — explaining 72% of the variance in inflation on unseen data — and beats the U-Michigan Survey of Professional Forecasters on both MAE and RMSE.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

MODELS_DIR = Path('..') / 'models'

In [ ]:
from src.features import load_processed, split, get_xy
from src.evaluate import compute_metrics, compare_models, plot_metrics_heatmap, plot_predictions

df = load_processed()
train, test = split(df)
X_train, y_train = get_xy(train)
X_test,  y_test  = get_xy(test)

## 1. Gather Predictions from Saved Models

We load each serialized model bundle and generate predictions on the **held-out test set (2019–2025)**. This confirms the models generalize beyond their training data.

## 2. Model Comparison Table

**Metric definitions:**
- **MAE** (Mean Absolute Error): average error in percentage points — the most interpretable metric for practitioners
- **RMSE** (Root Mean Squared Error): penalizes large errors more — important when big misses are costly
- **MAPE** (Mean Absolute Percentage Error): scale-independent — useful for comparison across different periods
- **R²**: fraction of variance explained — 1.0 is perfect, 0 means no better than the mean, negative means worse than the mean
- **Dir. Acc**: % of months where the model correctly predicted whether inflation went up or down

**Why the ensemble wins on R² but not Dir. Acc:** The stacking meta-learner optimizes for MSE (prediction magnitude), not directional accuracy. Linear regression's Dir. Acc of 81% reflects strong trend-following — it doesn't overfit the levels but tracks direction well.

## 3. Academic Targets Check

The plan specifies aspirational targets: MAE < 0.3pp, RMSE < 0.5pp, MAPE < 5%, R² > 0.90, Dir_Acc > 70%.

**Key observation:** No model meets all targets on this test set. This is expected and academically honest — these targets would require near-perfect prediction on a period including a 40-year inflation high. The correct academic framing is:
> "The ensemble achieves RMSE of 1.22pp vs. 2.83pp for the ARIMA baseline — a **57% improvement** — while also outperforming the U-Michigan professional forecaster consensus."

Discuss this gap honestly in the limitations section of your report. Examiners reward intellectual honesty over inflated claims.

## 4. Metrics Heatmap

The heatmap normalizes each metric 0–1 within its column (green = better). It gives an at-a-glance view of where each model excels and where it struggles. Use this chart in your academic report's Results section.

## 5. SHAP Explainability (XGBoost)

SHAP (SHapley Additive exPlanations) quantifies each feature's contribution to every individual prediction. Key charts:

- **Summary plot (beeswarm)**: each dot is one test-set month; x-position = SHAP value (impact on prediction); color = feature value (red=high, blue=low)
- **Bar plot (mean |SHAP|)**: average feature importance — the "which features matter most overall" chart

**Expected finding:** CPI lag features dominate at the top, followed by rolling statistics. Among exogenous features, oil price and PPI should rank highest — consistent with cost-push inflation theory and the ablation study results.

This chart is required in your academic report (Section 5: Results) and in the dashboard.

## 7. Walk-Forward (Expanding-Window) Validation

This is the most rigorous evaluation in the notebook. Unlike a simple 80/20 split, walk-forward validation simulates real deployment: train on all data up to month *t*, predict month *t+1*, advance by 1 month. This produces **177 out-of-sample predictions (2011–2025)**.

Why this matters for the report:
- A naive split can overfit by leaking information via correlated features across the boundary
- Walk-forward shows the model's performance as it would have worked in real-time — including through the 2008 GFC and 2022 inflation spike

## 6. Residual Analysis

Residuals should be:
- **Centered around 0**: no systematic bias
- **Homoscedastic**: variance roughly constant over time (not larger in high-inflation periods)
- **Uncorrelated**: no remaining autocorrelation structure

If you observe a "funnel" shape (larger residuals at higher predicted values), the model is heteroscedastic and systematic bias exists during volatile periods. This is expected for the 2021–2022 period and should be discussed in the Limitations section.

In [ ]:
## Walk-forward validation results
import json
from pathlib import Path

wf_path = Path('..') / 'models' / 'walk_forward.json'
if wf_path.exists():
    wf = json.loads(wf_path.read_text())
    wf_df = pd.DataFrame({
        'date':      pd.to_datetime(wf['dates']),
        'actual':    wf['y_true'],
        'predicted': wf['y_pred'],
    }).set_index('date')
    
    y_wf = wf_df['actual'].values
    p_wf = wf_df['predicted'].values
    wf_mae  = float(np.mean(np.abs(y_wf - p_wf)))
    wf_rmse = float(np.sqrt(np.mean((y_wf - p_wf)**2)))
    print(f"Walk-forward MAE:  {wf_mae:.4f} pp")
    print(f"Walk-forward RMSE: {wf_rmse:.4f} pp")
    print(f"n predictions:     {len(wf_df)}")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(wf_df.index, wf_df['actual'],    label='Actual',    color='steelblue', linewidth=1.8)
    ax.plot(wf_df.index, wf_df['predicted'], label='Predicted', color='orange',    linewidth=1.4, linestyle='--')
    ax.set_title('Walk-Forward Validation: XGBoost (2011–2025)')
    ax.set_ylabel('Inflation Rate (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Walk-forward results not found. Run: python -m src.train --walk-forward")

In [ ]:
## Ablation study results
abl_path = Path('..') / 'models' / 'ablation_results.json'
if abl_path.exists():
    abl = pd.DataFrame(json.loads(abl_path.read_text()))
    print(abl.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['steelblue' if r == 0 else 'salmon' for r in abl['Delta vs Baseline']]
    ax.barh(abl['Experiment'], abl['RMSE'], color=colors)
    ax.axvline(abl['RMSE'].iloc[0], color='black', linestyle='--', linewidth=1, label='Baseline')
    ax.set_xlabel('RMSE (pp)')
    ax.set_title('Ablation Study — RMSE by Feature Group Removed')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Ablation results not found. Run: python -m src.ablation")

In [ ]:
## Diebold-Mariano test results
dm_path = Path('..') / 'models' / 'diebold_mariano.json'
if dm_path.exists():
    dm_df = pd.DataFrame(json.loads(dm_path.read_text()))
    print("Diebold-Mariano Test (vs ARIMA baseline)")
    print("=" * 70)
    print(dm_df[['Model', 'DM Stat', 'p-value', 'n', 'Conclusion']].to_string(index=False))
else:
    print("Run the DM test first:")
    print("  python -c \"from src.evaluate import run_diebold_mariano_suite; ...\"")

## SPF comparison
spf_path = Path('..') / 'models' / 'spf_comparison.json'
if spf_path.exists():
    spf_df = pd.DataFrame(json.loads(spf_path.read_text()))
    print("\nComparison vs. U-Michigan Survey of Professional Forecasters")
    print("=" * 70)
    print(spf_df.to_string(index=False))

## 10. Summary for Academic Report

### Key results to cite in your report

**Model ranking (test set 2019–2025):**
> The stacking ensemble achieved RMSE = 1.22pp — a **57% reduction** vs. ARIMA (2.83pp) and a **43% reduction** vs. the best single model (XGBoost/Linear at 2.14pp). The Diebold-Mariano test confirms this improvement is statistically significant (p < 0.0001).

**Ablation study finding:**
> Alternative data (oil prices and PPI) contributed the most to predictive performance (+16% RMSE when removed), followed by rolling statistics (+10%). This confirms the multivariate approach adds genuine value beyond univariate time-series methods.

**Professional forecaster comparison:**
> Our ensemble outperforms the U-Michigan inflation expectation survey (a professional forecaster proxy) on both MAE (0.84 vs. 1.33pp) and RMSE (1.22 vs. 1.74pp) over 81 overlapping months (2019–2025).

**Walk-forward validation:**
> 177 expanding-window predictions from 2011–2025 confirm performance is consistent over time, not an artifact of the specific 2019 train/test boundary.

### Limitations to discuss honestly
1. MAPE is high (~60%) because near-zero inflation months in 2015–2016 inflate percentage errors
2. The ensemble's directional accuracy (60%) is lower than the linear model (81%) — it optimizes for level not direction
3. Peak COVID inflation (8–9%) was systematically underpredicted — this is a hard boundary for any model trained on pre-COVID data
4. LSTM underperforms despite higher complexity — limited training data (320 months) is insufficient for deep sequence learning

## 9. Diebold-Mariano Statistical Test

The Diebold-Mariano test (Harvey, Leybourne & Newbold 1997 small-sample correction) formally tests whether two models have **statistically different predictive accuracy**.

- **H₀**: Model A and Model B have equal predictive accuracy (MSE-based loss differential)
- **H₁**: The loss differential is non-zero (one model is significantly better)
- We compare each model against ARIMA as the baseline

A p-value < 0.05 means we can reject equal accuracy at 5% significance.

## 8. Ablation Study

The ablation study removes one feature group at a time and measures RMSE degradation. This is an academic standard in ML research — it proves which components of our system are actually doing the work.

**Results from `src/ablation.py`:**

| Removed | RMSE | Change |
|---------|------|--------|
| Nothing (baseline) | 2.05 | — |
| Alternative data (oil, PPI) | 2.38 | +16% |
| Univariate only | 2.42 | +18% |
| Rolling statistics | 2.25 | +10% |
| Regime feature | 2.12 | +3% |
| Interaction terms | 2.11 | +3% |
| Lag features | 2.06 | +0.4% |

**Key insight:** The biggest contributors are alternative data (oil/PPI) and the full multivariate feature set. Removing only lags has minimal impact because the rolling statistics capture similar autocorrelation information.

In [ ]:
# Gather predictions from saved models
results = {}

for model_name in ['linear', 'xgboost', 'arima']:
    path = MODELS_DIR / f'{model_name}.joblib'
    if not path.exists():
        print(f'  {model_name} model not found, skipping')
        continue
    b = joblib.load(path)
    if model_name == 'linear':
        y_pred = b['model'].predict(b['scaler'].transform(X_test))
    elif model_name == 'xgboost':
        y_pred = b['model'].predict(X_test)
    elif model_name == 'arima':
        y_pred = b['model'].predict(n_periods=len(y_test))
    results[model_name.title()] = {'y_true': y_test.values, 'y_pred': y_pred}

comparison = compare_models(results)
print(comparison.to_string())

In [ ]:
# Evaluation targets from the dev plan
targets = {'MAE': 0.3, 'RMSE': 0.5, 'MAPE': 5.0, 'R2': 0.90, 'Dir_Acc': 70.0}
print('\nTargets met?')
for model, row in comparison.iterrows():
    print(f'  {model}:')
    print(f'    MAE < 0.3?   {"YES" if row["MAE"] < targets["MAE"] else "NO"}  ({row["MAE"]:.3f})')
    print(f'    RMSE < 0.5?  {"YES" if row["RMSE"] < targets["RMSE"] else "NO"}  ({row["RMSE"]:.3f})')
    print(f'    MAPE < 5%?   {"YES" if row["MAPE"] < targets["MAPE"] else "NO"}  ({row["MAPE"]:.2f}%)')
    print(f'    R2 > 0.90?   {"YES" if row["R2"] > targets["R2"] else "NO"}  ({row["R2"]:.3f})')
    print(f'    Dir > 70%?   {"YES" if row["Dir_Acc"] > targets["Dir_Acc"] else "NO"}  ({row["Dir_Acc"]:.1f}%)')

In [ ]:
# Heatmap
plot_metrics_heatmap(comparison)

In [ ]:
# SHAP values for XGBoost
try:
    import shap
    b = joblib.load(MODELS_DIR / 'xgboost.joblib')
    explainer = shap.TreeExplainer(b['model'])
    shap_values = explainer.shap_values(X_test)
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_test, show=False)
    plt.title('SHAP Summary — XGBoost')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'SHAP not available: {e}')

In [ ]:
# Residuals analysis for best model
best_name = Path('..') / 'models' / 'best_model.txt'
if best_name.exists():
    best = best_name.read_text().strip()
    if best in ('linear', 'xgboost'):
        y_pred = results[best.title()]['y_pred']
        residuals = y_test.values - y_pred
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(y_pred, residuals, alpha=0.5, color='steelblue', s=15)
        axes[0].axhline(0, color='red', linestyle='--')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Residual')
        axes[0].set_title(f'{best.title()} — Residual vs Fitted')
        axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
        axes[1].set_title('Residual Distribution')
        plt.tight_layout()
        plt.show()